In [ ]:
import asyncio
import os
import gradio as gr
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent


load_dotenv()


In [ ]:
load_dotenv(override=True)

In [ ]:
class TicketAnalysis(BaseModel):
    category:  str = Field(description="billing, technical, shipping, refund, or general")
    urgency:   int = Field(description="Urgency score from 1 (low) to 5 (critical)")
    sentiment: str = Field(description="angry, frustrated, neutral, or satisfied")
    summary:   str = Field(description="One sentence summary of the core issue")

In [ ]:
class RoutingDecision(BaseModel):
    team: str = Field(description="Team to handle: billing_team, tech_support, logistics, customer_success")
    reason: str = Field(description="One sentence reason for this routing decision")
    escalate: bool = Field(description="True if this needs immediate escalation to a manager")

In [ ]:
analyser_agent = Agent(
    name="Ticket Analyser",
    instructions=(
        "You analyse customer support tickets. "
        "Categorise the ticket, score urgency 1-5, assess sentiment, "
        "and provide a one sentence summary of the core issue. "
        "Be precise and consistent."
    ),
    model="gpt-4o-mini",
    output_type=TicketAnalysis,
)

In [ ]:
response_agent = Agent(
    name="Response Drafter",
    instructions=(
        "You draft professional, empathetic customer support replies. "
        "Acknowledge the issue, provide a clear resolution path, "
        "and end with a positive closing. "
        "Keep responses under 150 words. "
        "Do not use placeholders like [NAME] - write naturally. "
        "Reply with ONLY the email draft, nothing else."
    ),
    model="gpt-4o-mini",
)

In [ ]:
router_agent = Agent(
    name="Ticket Router",
    instructions=(
        "You decide which team should handle a support ticket. "
        "billing_team: payment, invoice, subscription issues. "
        "tech_support: bugs, errors, login, product not working. "
        "logistics: shipping, delivery, tracking, damaged goods. "
        "customer_success: general questions, feedback, feature requests. "
        "Escalate (escalate=True) if urgency is 4 or 5, or sentiment is angry."
    ),
    model="gpt-4o-mini",
    output_type=RoutingDecision,
)

In [ ]:
async def run_pipeline(ticket: str):
    """
    Runs all three agents:
    - Analyser and Response drafter run IN PARALLEL (independent)
    - Router runs AFTER analyser (needs category/urgency to decide)
    """
    with trace("Support ticket pipeline"):
 
        # Step 1 — analyse and draft response IN PARALLEL
        # These are independent — neither needs the other's output
        analysis_task = asyncio.create_task(
            Runner.run(analyser_agent, ticket)
        )
        response_task = asyncio.create_task(
            Runner.run(response_agent, f"Customer ticket:\n{ticket}")
        )
 
        analysis_result, response_result = await asyncio.gather(
            analysis_task, response_task
        )
 
        analysis: TicketAnalysis = analysis_result.final_output
        draft: str = response_result.final_output
 
        # Step 2 — route AFTER analysis (needs category + urgency)
        router_input = (
            f"Ticket: {ticket}\n"
            f"Category: {analysis.category}\n"
            f"Urgency: {analysis.urgency}\n"
            f"Sentiment: {analysis.sentiment}"
        )
        routing_result = await Runner.run(router_agent, router_input)
        routing: RoutingDecision = routing_result.final_output
 
    return analysis, draft, routing

In [ ]:
SAMPLE_TICKETS = [
    "I've been charged twice for my subscription this month! This is completely unacceptable. I need an immediate refund or I'm cancelling and disputing the charge with my bank.",
    "Hi, I'm having trouble logging into my account. It keeps saying my password is wrong but I just reset it. Can you help?",
    "My order #45821 was supposed to arrive 3 days ago and still hasn't shown up. The tracking page just says 'in transit'.",
    "I love your product! I was wondering if you plan to add dark mode in a future update?",
]

In [ ]:
URGENCY_COLOURS = {1: "🟢", 2: "🔵", 3: "🟡", 4: "🟠", 5: "🔴"}
SENTIMENT_EMOJI = {"angry": "😡", "frustrated": "😤", "neutral": "😐", "satisfied": "😊"}

In [ ]:
def format_analysis(analysis: TicketAnalysis) -> str:
    urgency_dot = URGENCY_COLOURS.get(analysis.urgency, "⚪")
    sentiment_icon = SENTIMENT_EMOJI.get(analysis.sentiment, "❓")
    return (
        f"**Category:** {analysis.category.replace('_', ' ').title()}\n\n"
        f"**Urgency:** {urgency_dot} {analysis.urgency}/5\n\n"
        f"**Sentiment:** {sentiment_icon} {analysis.sentiment.title()}\n\n"
        f"**Summary:** {analysis.summary}"
    )

In [ ]:
def format_routing(routing: RoutingDecision) -> str:
    escalation = "🚨 **ESCALATE TO MANAGER**" if routing.escalate else "✅ Standard handling"
    team = routing.team.replace('_', ' ').title()
    return (
        f"**Assigned Team:** {team}\n\n"
        f"**Reason:** {routing.reason}\n\n"
        f"{escalation}"
    )

In [ ]:
def process_ticket(ticket: str):
    """Gradio handler — runs async pipeline synchronously"""
    if not ticket.strip():
        return "Please enter a ticket.", "", ""
 
    try:
        analysis, draft, routing = asyncio.run(run_pipeline(ticket))
        return (
            format_analysis(analysis),
            draft,
            format_routing(routing),
        )
    except Exception as e:
        error = f"Error: {str(e)}"
        return error, error, error

In [ ]:
with gr.Blocks(title="Support Ticket Analyser") as ui:
 
    gr.Markdown("# 🎫 Customer Support Ticket Analyser")
    gr.Markdown("Paste a customer ticket below. Three agents run in parallel to analyse, draft a reply, and route it.")
 
    with gr.Row():
        with gr.Column(scale=1):
            ticket_input = gr.Textbox(
                label="Customer Ticket",
                placeholder="Paste the customer's message here...",
                lines=8,
            )
            with gr.Row():
                analyse_btn = gr.Button("Analyse Ticket", variant="primary")
                clear_btn   = gr.Button("Clear")
 
            gr.Markdown("**Sample tickets:**")
            for i, sample in enumerate(SAMPLE_TICKETS):
                gr.Button(f"Sample {i+1}: {sample[:40]}...").click(
                    fn=lambda s=sample: s,
                    outputs=ticket_input
                )
 
        with gr.Column(scale=1):
            analysis_output = gr.Markdown(label="Ticket Analysis")
            gr.Markdown("---")
            routing_output  = gr.Markdown(label="Routing Decision")
 
    draft_output = gr.Textbox(
        label="Drafted Response",
        lines=8,
        interactive=True,           # agent editable by human before sending
    )

    analyse_btn.click(
        fn      = process_ticket,
        inputs  = [ticket_input],
        outputs = [analysis_output, draft_output, routing_output],
    )
    clear_btn.click(
        fn      = lambda: ("", "", "", ""),
        outputs = [ticket_input, analysis_output, draft_output, routing_output],
    )
    

In [ ]:
if __name__ == "__main__":
    ui.launch(inbrowser=True)